# Snippet from Math-SPD-Metrics-and-Riemannian-Geometry.md


In [ ]:
import numpy as np
from scipy.linalg import cholesky, eigh

class SymbolicManifoldMetric:
    """
    SPD metric for task-aware distance computation.
    
    Implements low-rank regularized SPD matrices with Cholesky whitening
    for stable distance calculations on learned manifolds.
    """
    
    def __init__(self, L: np.ndarray, delta: float = 1e-3):
        """
        Initialize SPD metric.
        
        Args:
            L: Low-rank factor (D x r).
            delta: Regularization parameter for numerical stability.
        """
        self.L = L
        self.delta = delta
        self.D = L.shape[0]
        self.r = L.shape[1]
        self.M = self._build_metric()
    
    def _build_metric(self) -> np.ndarray:
        """Construct M = L L^T + delta I."""
        return self.L @ self.L.T + self.delta * np.eye(self.D)
    
    def distance(self, x: np.ndarray, y: np.ndarray) -> float:
        """
        Compute Mahalanobis distance between x and y.
        
        Args:
            x: First point (D-dimensional).
            y: Second point (D-dimensional).
        
        Returns:
            Warped distance d_M(x, y).
        """
        # Cholesky factorization for whitening
        try:
            W = cholesky(self.M, lower=False)
        except np.linalg.LinAlgError:
            # Fallback: increase regularization
            M_reg = self.M + self.delta * np.eye(self.D)
            W = cholesky(M_reg, lower=False)
        
        diff = x - y
        z = W @ diff  # Whitened difference
        return np.linalg.norm(z)
    
    def condition_number(self) -> float:
        """Compute condition number of M."""
        eigenvalues = np.linalg.eigvalsh(self.M)
        return eigenvalues.max() / eigenvalues.min()
    
    @staticmethod
    def low_rank_fit(X: np.ndarray, r: int = 4) -> np.ndarray:
        """
        Learn low-rank factor L from data covariance.
        
        Args:
            X: Data matrix (n x D).
            r: Target rank.
        
        Returns:
            Low-rank factor L (D x r).
        """
        # Compute sample covariance
        C = np.cov(X.T)
        
        # Eigendecomposition and truncation
        eigenvalues, eigenvectors = eigh(C)
        
        # Select top r eigenvalues (sorted ascending by eigh)
        top_r_indices = np.argsort(eigenvalues)[-r:][::-1]
        top_eigenvalues = eigenvalues[top_r_indices]
        top_eigenvectors = eigenvectors[:, top_r_indices]
        
        # Build L = U_r sqrt(Lambda_r)
        L = top_eigenvectors @ np.diag(np.sqrt(top_eigenvalues))
        
        return L

# Example Usage: Embed and Measure
np.random.seed(42)

# Generate synthetic embeddings
D = 35  # Dimension
r = 4   # Rank
n_samples = 100

# Create random low-rank factor
L = np.random.randn(D, r)
metric = SymbolicManifoldMetric(L, delta=1e-3)

# Sample points
x = np.random.randn(D)
y = x + 0.1 * np.random.randn(D)  # Nearby point

# Compute distances
d_M = metric.distance(x, y)
d_euclidean = np.linalg.norm(x - y)

print("SPD Metric Distance Computation")
print("=" * 50)
print(f"Dimension D: {D}")
print(f"Rank r: {r}")
print(f"Regularization delta: {metric.delta}")
print(f"\nMahalanobis distance d_M: {d_M:.4f}")
print(f"Euclidean distance: {d_euclidean:.4f}")
print(f"Ratio d_M / d_euclidean: {d_M / d_euclidean:.4f}")
print(f"\nCondition number of M: {metric.condition_number():.2e}")

# Fit from data example
print("\n" + "=" * 50)
print("Low-Rank Fitting Example")
print("=" * 50)

X = np.random.randn(n_samples, D)
L_fitted = SymbolicManifoldMetric.low_rank_fit(X, r=r)
metric_fitted = SymbolicManifoldMetric(L_fitted, delta=1e-3)

print(f"Fitted L shape: {L_fitted.shape}")
print(f"Fitted M condition number: {metric_fitted.condition_number():.2e}")
